## Integración con Langchain y langgraph

In [1]:
from dotenv import load_dotenv
import os

import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from openai import OpenAI

import ast

from IPython.display import HTML, Markdown, display
import pandas as pd

# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [2]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [3]:
# clientes para encoding de query, etc

client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [30]:
import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}

pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']
mkdown_tables = los_mex_dict['mkdown_tables']

los_mex_dict cargado --  leer readme_dict para info


In [5]:
# carga de db_f1 en chroma

DB_PERSIST_DIR = "./db_f1"

client = chromadb.PersistentClient(
    path=DB_PERSIST_DIR,
    settings=Settings(),
    tenant=DEFAULT_TENANT,        # usually "default_tenant"
    database=DEFAULT_DATABASE     # this is "chroma" by default
)                                 # :contentReference[oaicite:0]{index=0}

# see what collections you have
print("Collections:", [c.name for c in client.list_collections()])

# pick the one you need (for example "enc_dbf1" *if* that was your collection name)
db_f1 = client.get_collection(name="enc_dbf1")
db_f1.peek()


Collections: ['enc_dbf1']


{'ids': ['p1_1|IDE__question',
  'p1_1|IDE__summary',
  'p1_1|IDE__implications',
  'p2_1|IDE__question',
  'p2_1|IDE__summary',
  'p2_1|IDE__implications',
  'p3_1|IDE__question',
  'p3_1|IDE__summary',
  'p3_1|IDE__implications',
  'p3_2|IDE__question'],
 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
         -0.01462473, -0.00672276],
        [-0.00439202, -0.005377  ,  0.03002077, ..., -0.00975933,
         -0.02509912, -0.04386856],
        [-0.03001864,  0.00425975,  0.03407801, ...,  0.00466375,
         -0.01384581, -0.03074261],
        ...,
        [ 0.02034639,  0.01068023,  0.01885129, ..., -0.00335098,
         -0.01566607, -0.04771326],
        [-0.01296452,  0.00133349,  0.02093472, ..., -0.00910233,
         -0.00846187, -0.02445403],
        [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
         -0.01164658, -0.00516163]], shape=(10, 1536)),
 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio comida, mercado, 

### query v2: creación de resúmenes temáticos

In [6]:
# query de la base de embeddings
# query_emb es el embedding de la pregunta

user_query= '¿qué significa ser mexicano para los mexicanos?'
# turn it into a vector
query_emb = embedding_fun_openai([user_query])[0]
query_emb

array([-0.01168481, -0.026932  ,  0.01919517, ...,  0.00275288,
       -0.02104368, -0.01839945], shape=(1536,), dtype=float32)

In [7]:
# ejemplo: query entre las preguntas

result_q = db_f1.query(
    query_embeddings = [query_emb],
    n_results        = 5,
    where            = {"type": "question"}
)

print(result_q["documents"])   # the matched question strings
print(result_q["metadatas"])   # metadata for each hit (includes qid & type)


[['CULTURA_POLITICA|¿Qué tan orgulloso se siente de ser mexicano?', 'IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano?', 'INDIGENAS|¿Cuál cree que es la mayor ventaja de ser indígena en México?', 'INDIGENAS|¿Y cuál cree que es la mayor desventaja de ser indígena en México?', 'IDENTIDAD_Y_VALORES|Y Ahora dígame; en su opinión ¿qué tanto los mexicanos mienten para obtener un beneficio?']]
[[{'qid': 'p5|CUL', 'type': 'question'}, {'type': 'question', 'qid': 'p6|IDE'}, {'type': 'question', 'qid': 'p13|IND'}, {'qid': 'p14|IND', 'type': 'question'}, {'qid': 'p46_4|IDE', 'type': 'question'}]]


In [8]:
# tmp_dist_df contiene las distancias para los tres tipos/ facets, normalizadas entre 0 y 1

# OJO: esto obviamente asume que los tres tipos de información tienen la misma importancia, lo cual no es inicialmente cierto.
# Pero la mezcla de las tres calificaciones devuelve una variedad más amplia de resultados.  

type_lst = [ "question", "summary", "implications"]

tmp_dist_dict = {}
for type in type_lst:
    print(f"Querying for type: {type}")
    tmp_res_q = db_f1.query(
        query_embeddings = [query_emb],
        n_results        = 100,  # devuelvo 100 resultados para cada tipo con distancias menores
        where            = {"type": type}
    )
    [tmp_res_ids] = tmp_res_q['ids']
    [tmp_res_distances] = tmp_res_q['distances']

    tmp_dist_dict[type]= dict(zip(tmp_res_ids, tmp_res_distances))

# remove the suffixes from the keys
tmp_dist_dict = { outer_key: { k.split('__')[0]: v for k, v in inner_dict.items() }
    for outer_key, inner_dict in tmp_dist_dict.items() }

# Create a DataFrame where keys in every subdict are the index and keys in tmp_dist_dict are columns
tmp_dist_df = pd.DataFrame.from_dict(tmp_dist_dict)

# Normalize each column so that max = 1 and min = 0
tmp_dist_df = (tmp_dist_df - tmp_dist_df.min()) / (tmp_dist_df.max() - tmp_dist_df.min())

Querying for type: question
Querying for type: summary
Querying for type: implications


In [9]:
# tmp_res_st contiene las 40 preguntas más cercanas y la descripción de sus resultados.

top_vals = 40

tmp_dist_df['mean'] = tmp_dist_df.mean(axis=1)
tmp_dist_df.sort_values(by='mean', ascending=True, inplace=True)

top_ids = tmp_dist_df.head(top_vals).index.tolist()

tmp_list = []

for type in type_lst:
    for id in top_ids:
        tmp_list.append(id + f'__{type}')


# Retrieve documents using the list of ids
result_by_ids = db_f1.get(ids=tmp_list)

tmp_pre_res_dict = dict(zip(result_by_ids['ids'], result_by_ids['documents']))

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# # instead of your current list-comp do this:
tmp_combined_strings = []

 # Calculate the group index based on the position

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

'QUESTION_1|p1_1|IDE: Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN\nSUMMARY_1|p1_1|IDE: The table presents responses to the question: \'Please give three words you associate with Mexico,\' with the answer options not explicitly listed but represented as \'nan\' (no answer). The percentages vary widely, with some high values such as 12.83%, 4.42%, and 4.00%, indicating notable proportions of respondents selecting those options. Several smaller percentages range from about 0.08% to around 2.5%, reflecting diverse associations or lack of responses. The high \'no answer\' or \'did not know\' options (some over 12%) suggest a significant portion of participants either did not respond or were uncertain, which may imply a divergence in perceptions or a lack of strong associations with Mexico.\nQUESTION_2|p2_1|IDE: Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXIC

In [10]:

# generación de prompt para seleccionar analizar las tablas generadas - v2
# generación aumentada: prompt + pregunta + documentos 

def create_prompt_crosssum(user_query, tmp_res_st, n_topics= 5):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

    Your taks is to read the survey RESULTS and find the top {n_topics} SIMILAR PATTERNS across the RESULTS that best answer the QUERY below.
    Note SIMILAR PATTERNS are those that show agreement across different questions and summaries, and that are relevant to the QUERY. 
    You will also retrun {n_topics} DIFFERENT PATTERN that is relevant to the QUERY, but that shows a pattern that contradicts any of the SIMILAR PATTERNS.

    Note that the RESULTS will have QUESTION_IDS of the format 'question_id|table_id') (eg |pxx|TYY where x is a number and YY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS in your RETURN OBJECT. 

    For example, in the following results, 

    for TEST_QUERY: 'Do people like sweets?'
    the TEST_RESULTS are:

    QUESTION_1|p1_1|TPC1: 'Do you like ice cream?'
    SUMMARY_1|p1_1|TPC1: 85% of people like ice cream, while 10% said they do not like it, and 5% said they do not know.

    QUESTION_2|p2|TPC1: 'Do you like marshmallow treats?'
    SUMMARY_2|p2|TPC1: 80% of people like marshmallow treats, while 15% said they do not like it, and 5% said they do not know.
    
    QUESTION_3|p10|TPC2: 'Do you like sour apple candies?'
    SUMMARY_3|p10|TPC2: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

    In these cases, the SIMILAR PATTERN is that 'people really like sweat treats because 85% like ice cream and 80% like marshmallow treats', 
    and the DIFFERENT PATTERN is that 'people do not like sour treats as much beacuse only 40% said they liked them'.

    Note that any mention of results not available or 'NaN' is NOT THE SAME as 'Do now know' or 'no answer'. 
    You should ignore any reference to results that are not available or 'NaN'.
    
    If you discuss two similar cateries together (eg, 'like a lot' and 'like somehow'), only  mention the sum of these percentages if you have the all the original ones to sum them. Otherwise, do not mention the figure itself. 
    Do not discuss figures or percentages you are not sure about.

    Be careful to not be repetitive and to add a varied list of PATTERNS: if you have discussed any SIMILAR PATTERN or DIFFERENT PATTERN, do not add it to the list again, but come up with a different one. 
    
    QUERY: {user_query}
    RESULTS: {tmp_res_st.replace("\n", " ")}

    For every SIMILAR PATTERN or DIFFERENT PATTERN you identify, you will write a summary TITLE SUMMARY of the topic of 3 lowercase words separated by '_' (eg, 'people_like_sweets').
    For every SIMILAR PATTERN or DIFFERENT PATTERN you identify, you will keep track of the QUESTION IDS of the questions that are you used to create them and refer to them and you will write them as a VARIABLE_STRING with format |pxx|TYY (where x is a number and YY is a table id), but separating using ',' eg 'p1_1|TPC1,p2|TPC1,p10|TPC2'.

    You will return a RETURN OBJECT that will be a python dict SIMILAR_PATTERN_1, SIMILAR_PATTERN_2, ..., SIMILAR_PATTERN_{n_topics} and DIFFERENT_PATTERN_1, DIFFERENT_PATTERN_2, ..., DIFFERENT_PATTERN_{n_topics}. 
    Inside each of these keys, a second-level dict will have the keys TITLE_SUMMARY where your TITLE SUMMARY will be stored as a value, VARIABLE_STRING where the VARIABLE_STRING will be stored as a value, and DESCRIPTION where you will write the description of each pattern.
    These DESCRIPTIONS should describe the pattern and explain it with relevant numbres and explanations. 

    Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.
    Do not add any other text to it.
    """
    return prompt

# Example usage:
prompt = create_prompt_crosssum(user_query, tmp_res_st)

# update get_answer to accept and pass a temperature parameter
def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    """Get a simple answer from OpenAI models without chatbot functionality."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


mod_alto = 'gpt-4.1-2025-04-14' # toma ~30 segs para responder correctamente
mod_bajo = 'gpt-4.1-nano-2025-04-14' # no puede
mod_med = 'gpt-4.1-mini-2025-04-14' # no puede



tmp_ans = get_answer(prompt= prompt, system_prompt=None, model = mod_alto)

tmp_ans_dict = ast.literal_eval(tmp_ans)

tmp_ans_dict

{'SIMILAR_PATTERN_1': {'TITLE_SUMMARY': 'strong_national_pride',
  'VARIABLE_STRING': 'p3_5|IDE,p6|IDE,p5|CUL,p17_2|GLO,p18|GLO',
  'DESCRIPTION': "A strong majority of Mexicans express significant pride and emotional connection to both their national identity and to Mexican culture. For example, over 81% feel close to Mexico (p3_5|IDE), and more than 88% feel 'very' or 'somewhat' proud of being Mexican (p6|IDE, p5|CUL). Pride is especially high in areas like Mexican food (67.25% 'very proud', p17_2|GLO) and history (76.25% 'very' or 'somewhat' proud, p18|GLO), reflecting a deep-rooted, positive collective identity."},
 'SIMILAR_PATTERN_2': {'TITLE_SUMMARY': 'cultural_diversity_recognized',
  'VARIABLE_STRING': 'p25|DEP,p5|IDE',
  'DESCRIPTION': "Mexicans largely acknowledge the diversity within their national identity. 67.08% prefer to talk about 'the Mexican cultures' rather than a single homogenous culture (p25|DEP), and many identify both as Mexican and with their regional identiti

In [13]:
# langchain y pydantic 

import os
from typing import Dict
from pydantic import RootModel
from langchain.output_parsers import PydanticOutputParser
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI

# 1. Define a Pydantic model for a nested dict:
#    Outer dict: str → inner dict
#    Inner dict: str → str
class ResponseModel(RootModel[Dict[str, Dict[str, str]]]):
    """
    Root model: mapping from facet group names (str) to dictionaries mapping str to str.
    """
    pass

# 2. Create a PydanticOutputParser using that model
parser = PydanticOutputParser(pydantic_object=ResponseModel)
format_instructions = parser.get_format_instructions()


# 3. Create a prompt template that uses the parser
prompt_template = PromptTemplate(
    input_variables=["user_query", "tmp_res_st"],
    template=create_prompt_crosssum("{user_query}", "{tmp_res_st}"),
    partial_variables={"n_topics": 5}, 
)


prompt_template = PromptTemplate(
    input_variables=["user_query", "tmp_res_st"],
    template=(
        create_prompt_crosssum("{user_query}", "{tmp_res_st}")
        + "\n\n{format_instructions}"
    ),
    partial_variables={
        "n_topics": 5,
        "format_instructions": format_instructions,
    },
)

# 4. Define a helper that runs the chain and parses to our model
def get_structured_answer(
    user_query: str,
    tmp_res_st: str,
    model_name: str = "gpt-4o-mini-2024-07-18", # ?? funcionará con el mini?? 
    temperature: float = 0.9
) -> Dict[str, Dict[str, str]]:
    """
    Returns a nested dict: { facet_group: { key: value, ... }, ... }
    """
    llm = ChatOpenAI(
        model_name=model_name,
        temperature=temperature,
        openai_api_key=os.environ["OPENAI_API_KEY"]
    )
    chain = LLMChain(
        llm=llm,
        prompt=prompt_template,
        output_parser=parser,
        verbose=False
    )
    # new: call() will run LLM → parser.parse → give back your Pydantic model
    result = chain.invoke({
        "user_query": user_query,
        "tmp_res_st": tmp_res_st,
    })
    response_model: ResponseModel = result["text"]

    # Return the underlying dict-of-dicts
    return response_model.root

# 5. Call the function
user_query = '¿qué significa ser mexicano para los mexicanos?'

tst_lgc_dict = get_structured_answer(
    user_query=user_query,
    tmp_res_st=tmp_res_st,
    model_name=mod_alto, # mini produce resultados consistentes, aunque toma entre 30 y 60 segs. Nano hace una selección rápida, pero más superficial. 
    temperature=0.9
)

tst_lgc_dict

{'SIMILAR_PATTERN_1': {'TITLE_SUMMARY': 'national_pride_identity',
  'VARIABLE_STRING': 'p3_5|IDE,p4|IDE,p5|IDE,p8|CUL,p18|GLO,p20|GLO',
  'DESCRIPTION': "A consistent majority of respondents report strong national pride and identification with being Mexican. For example, 81.25% feel some degree of closeness to Mexico (p3_5|IDE), over 88% feel a lot or some pride in being Mexican (p4|IDE), and 67.25% are very proud of Mexican food (p20|GLO). Nearly half identify solely as Mexican (p5|IDE), and high levels of pride are reported for aspects like Mexican history (76.25% 'very' or 'somewhat' proud, p18|GLO). This pattern demonstrates that being Mexican is strongly tied to pride and emotional identification with the country."},
 'SIMILAR_PATTERN_2': {'TITLE_SUMMARY': 'cultural_diversity_valued',
  'VARIABLE_STRING': 'p25|DEP,p2_1|IDE',
  'DESCRIPTION': "There is a clear preference for acknowledging Mexico's cultural diversity rather than a single homogenized identity. 67.08% of respondents 

In [ ]:
## test para prompt de fact checking e interpolación 

tmp_ky = 0
tmp_kys = list(tst_lgc_dict.keys())

ky = tmp_kys[tmp_ky]

# variables identificadas por el modelo
tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
tst_str_lst = [st + '__question' for st in tst_str_lst] #[st.split('__')[0] for st in tst_str_lst]
print(f'variables identificadas por el modelo: {tst_str_lst}')

# variables en la base
tmp_db_var_lst= db_f1.get(
    ids=tst_str_lst)['ids']
print(f'variables en la base: {tmp_db_var_lst}')

# variables alucinadas
tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
if tmp_hlc_var_lst:
    print(f'variables alucinadas por el modelo: {tmp_hlc_var_lst}')

# implicaciones para las variables identificadas

tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]

tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]

tmp_imp_dict = dict(
    zip(
    tmp_ky_lst, 
    db_f1.get(ids=tmp_imp_lst)['documents']
    )
)

# resumenes 

tmp_smry_lst = [st.replace('question', 'summary') for st in tmp_db_var_lst]

tmp_smry_dict = dict(
    zip(
    tmp_ky_lst, 
    db_f1.get(ids=tmp_smry_lst)['documents']
    )
)

tmp_exp_ky = 0

print(f'implication in tmp_imp_dict: {tmp_imp_dict[tmp_ky_lst[tmp_exp_ky]]}')
print( f'summary in tmp_smry_dict: {tmp_smry_dict[tmp_ky_lst[tmp_exp_ky]]}')


# TODO: extraer ids que sí aparezcan, flag los que no, match de los que aparezcan con mkdown_tables, construir prompt para fact checking que:
#  asegure que el resultado es correcto y flag a los que no sean, y devuelva descripción revisada. 


variables identificadas por el modelo: ['p3_5|IDE__question', 'p4|IDE__question', 'p5|IDE__question', 'p8|CUL__question', 'p18|GLO__question', 'p20|GLO__question']
variables en la base: ['p3_5|IDE__question', 'p4|IDE__question', 'p8|CUL__question', 'p20|GLO__question']
variables alucinadas por el modelo: {'p18|GLO__question', 'p5|IDE__question'}
implication in tmp_imp_dict: As an expert in 'identity and values,' these results are crucial because they quantify the degree of national attachment among individuals, which is a fundamental aspect of understanding collective identity and societal cohesion. Other experts could utilize these findings to explore correlations between feelings of national belonging and other social or cultural values, helping to reveal how identity shapes behaviors, attitudes, and social integration. Recognizing the high level of national attachment can inform policies or initiatives aimed at strengthening cultural identity and fostering community solidarity.
summ

In [72]:
# prompt para análisis de interpolación por experto

def create_prompt_intpl(tmp_tpc, tmp_stmt, tmp_smry, tmp_mkd):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    prompt = f"""
    You are a very thorough research assistant and expert in {tmp_tpc} that is working on a survey research project.
    If your don't know what {tmp_tpc} means, translate it to English and then answer the question. 
    You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

    Your will have 2 tasks and will return 1 python dict with the results. 

    1) your first task is to fact-check the SUMMARY of survey results and then compare them with the table that will follow them. 
    If there are insconsitencies or errors in the numbers or explanations, you will correct the error and write a REVIEWED_SUMMARY.
    If the numbers are correct, you will make sure that explanations are clear and correct and make any adjustments into the REVIEWED_SUMMARY.

    here is the SUMMARY: {tmp_smry}.
    Here is the table of the survey results: {tmp_mkd}.

    Keep your REVIEWED_SUMMARY in mind for the next task. But you will not add it to the dict, only refer to it in the next task.

    2) Your second task is to read the STATEMENT from another expert in {tmp_tpc} about what he expects to find in the survey results and why they are important. 
    You will compare the STATEMENT with the REVIEWED_SUMMARY and write a one-sentence explanation of why the results in the REVIEWED_SUMMARY attend to the expectations of the STATEMENT.
    
    Here's the STATEMENT by another expert: {tmp_stmt}. 
    
    You will write this explanation in a python dict with the key 'EXPERT_SUMMARY_REVIEW' and the value will be the explanation.
    Be discriptive and include the most relevant numbers and explanations. Do not repeat the same information in different words, be aware of the expectations of the expert.
    Do not mention the exprert, the STATEMENT or the SUMARY. Do not add any new information that is not in the table.
    Do not mention survey questions, question ids, or the table. Only discuss the results.

    Be descriptive and include the most relevant numbers and explanations. Do not repeat the same information in different words, be aware of the expectations of the expert.
    Be careful to be clear what opinions you are talking about (ie, say '10.5% agree that icecream is good' instead of '10.5% agree that it is good').

    Here's the table of the survey results: {tmp_mkd}.

    You will write this explanation in a python dict with the key 'EXPERT_SUMMARY_REVIEW' and the value will be the explanation.

    Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.
    Do not add any other text to it.
    """
    return prompt


tmp_exp_ky = 0


tmp_tpc = rev_enc_nom_dict[tmp_ky_lst[tmp_exp_ky].split('|')[1]]
tmp_stmt = tmp_imp_dict[tmp_ky_lst[tmp_exp_ky]]
tmp_smry = tmp_smry_dict[tmp_ky_lst[tmp_exp_ky]]
tmp_mkd = mkdown_tables[tmp_ky_lst[tmp_exp_ky]]
# Example usage:
prompt = create_prompt_intpl(tmp_tpc, tmp_stmt, tmp_smry, tmp_mkd)

print(prompt)



    You are a very thorough research assistant and expert in POBREZA that is working on a survey research project.
    If your don't know what POBREZA means, translate it to English and then answer the question. 
    You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

    Your will have 2 tasks and will return 1 python dict with the results. 

    1) your first task is to fact-check the SUMMARY of survey results and then compare them with the table that will follow them. 
    If there are insconsitencies or errors in the numbers or explanations, you will correct the error and write a REVIEWED_SUMMARY.
    If the numbers are correct, you will make sure that explanations are clear and correct and make any adjustments into the REVIEWED_SUMMARY.

    here is the SUMMARY: The table shows that a majority of respondents, 60.42%, believe that in Mexico, all people are equal regardless of socio-economic status, while 36.92% th

In [105]:
## test para prompt de fact checking e interpolación 

tmp_fct_intp_dict = {}

for ky in tmp_kys:
    print(f'key: {ky}')
    # variables identificadas por el modelo
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst] #[st.split('__')[0] for st in tst_str_lst]
    print(f'variables identificadas por el modelo: {tst_str_lst}')

    # variables en la base
    tmp_db_var_lst= db_f1.get(
        ids=tst_str_lst)['ids']
    print(f'variables en la base: {tmp_db_var_lst}')

    # variables alucinadas
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'variables alucinadas por el modelo: {tmp_hlc_var_lst}')
        if not tmp_db_var_lst:
            print(f'no hay variables en la base para {ky}')
            continue        


    # implicaciones para las variables identificadas

    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]

    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]

    tmp_imp_dict = dict(
        zip(
        tmp_ky_lst, 
        db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    # resumenes 

    tmp_smry_lst = [st.replace('question', 'summary') for st in tmp_db_var_lst]

    tmp_smry_dict = dict(
        zip(
        tmp_ky_lst, 
        db_f1.get(ids=tmp_smry_lst)['documents']
        )
    )

    for i, var_ky in enumerate(tmp_ky_lst):

        print(f'key: {var_ky}\n')
        print(f'implication in tmp_imp_dict: {tmp_imp_dict[var_ky]}')
        print( f'summary in tmp_smry_dict: {tmp_smry_dict[var_ky]}')
        print('---')

        tmp_tpc = rev_enc_nom_dict[tmp_ky_lst[i].split('|')[1]]
        tmp_stmt = tmp_imp_dict[tmp_ky_lst[i]]
        tmp_smry = tmp_smry_dict[tmp_ky_lst[i]]
        tmp_mkd = mkdown_tables[tmp_ky_lst[i]]
        # Example usage:
        prompt = create_prompt_intpl(tmp_tpc, tmp_stmt, tmp_smry, tmp_mkd)

        tmp_ans = get_answer(prompt= prompt, system_prompt=None, model = mod_alto)

        tmp_fct_intp_dict[var_ky] = ast.literal_eval(tmp_ans)

        print(f'tmp_fct_intp_dict: {tmp_fct_intp_dict[var_ky]}')



# TODO: extraer ids que sí aparezcan, flag los que no, match de los que aparezcan con mkdown_tables, construir prompt para fact checking que:
#  asegure que el resultado es correcto y flag a los que no sean, y devuelva descripción revisada. 


key: SIMILAR_PATTERN_1
variables identificadas por el modelo: ['p3_5|IDE__question', 'p4|IDE__question', 'p5|IDE__question', 'p8|CUL__question', 'p18|GLO__question', 'p20|GLO__question']
variables en la base: ['p3_5|IDE__question', 'p4|IDE__question', 'p8|CUL__question', 'p20|GLO__question']
variables alucinadas por el modelo: {'p18|GLO__question', 'p5|IDE__question'}
key: p3_5|IDE

implication in tmp_imp_dict: As an expert in 'identity and values,' these results are crucial because they quantify the degree of national attachment among individuals, which is a fundamental aspect of understanding collective identity and societal cohesion. Other experts could utilize these findings to explore correlations between feelings of national belonging and other social or cultural values, helping to reveal how identity shapes behaviors, attitudes, and social integration. Recognizing the high level of national attachment can inform policies or initiatives aimed at strengthening cultural identity an

In [117]:
tmp_fct_intp_dict

{'p3_5|IDE': {'EXPERT_SUMMARY_REVIEW': "A combined 81.25% of individuals report feeling either 'much' or 'some' connection to Mexico (48.17% 'much' and 33.08% 'some'), while 13.67% feel 'little' connection and 4.67% feel 'none', with only 0.42% not responding, illustrating that a strong sense of national attachment is present among the vast majority and providing robust evidence of widespread collective identity and social cohesion, which is essential for understanding how national belonging may influence other social values and community solidarity."},
 'p4|IDE': {'EXPERT_SUMMARY_REVIEW': 'A majority of 55.25% believe it is possible to build a great nation even with diverse cultures and values, while 38.42% think a similar culture and shared values are necessary; the small percentages selecting "Other" (2.17%), "None" (1.83%), or "Don’t know/No response" (2.33%) reinforce the trend of greater support for pluralism and inclusivity, highlighting current societal attitudes that prioritiz

In [115]:
# prompt para análisis transversal por experto

def create_prompt_trnsvl(tmp_smry_st, user_query):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do its work in either language. 

    Your will have 3 tasks and will return 1 python dict with the results.

    1) your first task is to read the following list of statements made by experts in survey research, that mention results, their implications and their relevance.
    Note that each statement starts with marker * and contains all information for a single statement: results, implications and relevance. 
    You will three common topics across the statements and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each. 
    You will format your answer as a python dict with the names of the topics as keys in all caps and the summaries as values.

    Here are the statements: {tmp_smry_st}.

    You will add your summaries in a python dict with the key 'TOPIC_SUMMARIES'.

    2) You  will read your TOPIC_SUMMARIES and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    You will save your summary in a python dict with the key 'TOPIC_SUMMARY'.

    3) You will read the QUERY and your TOPIC_SUMMARY and write a two-sentence answer to the QUERY that summarizes the most important points of your TOPIC_SUMMARY. 
    Be concise and do not repeat numbers, just answer to the QUERY with the relevant ideas.
    Here is the QUERY: {user_query}.
    
    You will save your answer in a python dict with the key 'QUERY_ANSWER'.

    IMPORTANT: Your replies for all 3 tasks will be in the language of the QUERY.
    IMPORTANT: Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.
    Do not add any other text to it.
    """
    return prompt

tmp_exp_smry_dict = {ky: sdct['EXPERT_SUMMARY_REVIEW'] for ky, sdct in tmp_fct_intp_dict.items()}
tmp_smry_st= ' '.join([' * ' + st for st in tmp_exp_smry_dict.values()])
tmp_smry_st

# Example usage:
prompt = create_prompt_trnsvl(tmp_smry_st, user_query)

print(prompt)

tmp_ans = get_answer(prompt= prompt, system_prompt=None, model = mod_alto)

tmp_ans



    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do its work in either language. 

    Your will have 3 tasks and will return 1 python dict with the results.

    1) your first task is to read the following list of statements made by experts in survey research, that mention results, their implications and their relevance.
    Note that each statement starts with marker * and contains all information for a single statement: results, implications and relevance. 
    You will three common topics across the statements and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each. 
    You will format your answer as a python dict with the names of the topics as keys in all caps and the summaries as values.

    Here are the statements:  * A combined 81.25% of individuals report feeling either 'much' or 'some' connection to Mexico (48.17% 

"{'TOPIC_SUMMARIES': {'IDENTIDAD NACIONAL Y PERCEPCIONES SOBRE SER MEXICANO': 'Los resultados muestran que existe un fuerte sentido de conexión nacional entre los mexicanos, con más del 80% reportando sentirse muy o algo conectados con México, aunque al mismo tiempo predominan percepciones fragmentadas y diversas sobre los símbolos, asociaciones y rasgos que definen la identidad mexicana. Hay una marcada pluralidad y falta de consenso en torno a los significados del término “mexicano” y las asociaciones culturales, con altos niveles de respuestas dispersas o nulas y una baja identificación colectiva en torno a símbolos o valores únicos. La identificación con raíces indígenas, aunque significativa, también se distribuye de manera matizada, con proporciones similares entre quienes se identifican plenamente, parcialmente o nada, y una minoría con conexiones familiares con lenguas indígenas, ilustrando la complejidad y diversidad interna de la identidad nacional.', 'DIVERSIDAD, INCLUSIÓN Y

In [ ]:
# prompt para análisis transversal por experto

def create_prompt_plot_chose(p_id_sums_str, smry):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    prompt = f"""
    You are a very thorough research assistant and expert in survey research and data visualization.
    You are fully bilingual in English and Spanish, and can do its work in either language. 

    Your will have 1 task and will return 1 python dict with the results.

    1) your task is to read the following TOPIC_SUMMARY and then the list of pairs of QUESTION IDS AND SUMMARIES.
    Note that the QUESTION_ID is a string with the format 'pxx|TYY' where x is a number and YY is a table id, and the SUMMARY is separatged by a colon ':'
    (ie, 'pxx|TYY: summary of the question'). Note that the different pairs are separated by a '+'.
    The QUESTION ID is not relevant to your answer, but you will need to keep track of these QUESTION_IDS in your RETURN OBJECT.
    The SUMMARY describes the result of the survey question and you will compare each of them to the TOPIC_SUMMARY.

    Here is the TOPIC_SUMMARY: {smry}.
    Here are the pairs of QUESTION_ID and SUMMARY paris: {p_id_sums_str}.

    You have to choose the QUESTION_ID AND SUMMARY pair whose summary best ilustrates the TOPIC_SUMMARY.
    The evidence may not describe precisely waht the TOPIC_SUMMARY talks about, but it should be the one that best ilustrates the topic.

    You will return a python dict with the QUESTION_ID as the key and the SUMMARY as the value. 
    IMPORTANT: return the QUESTION_ID exactly as it is in the input, do not change it.
    IMPORTANT: Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.
    Do not add any other text to it.
    """
    return prompt




# string con todos los resultados identificados 
p_id_sums_str = ' '.join([' * ' + st for st in [': '.join([k, v]) for k, v in tmp_exp_smry_dict.items()]])

# diccionario con resumenes
smry_dict = ast.literal_eval(tmp_ans)


# diccionario para guardar las variables a graficasb
tmp_plot_chose_dict = {}

for ky, smry in smry_dict['TOPIC_SUMMARIES'].items():
    prompt = create_prompt_plot_chose(p_id_sums_str, smry)
    tmp_ans = get_answer(prompt= prompt, system_prompt=None, model = mod_alto)

    tmp_plot_chose_dict[ky] = ast.literal_eval(tmp_ans)

tmp_plot_chose_dict

{'IDENTIDAD NACIONAL Y PERCEPCIONES SOBRE SER MEXICANO': {'p2_1|IDE': 'The results show an exceptionally high proportion of non-responses or unspecified associations with the word "Mexicano," with notable figures such as 9.08%, 8.17%, 5.25%, 3.67%, and 2.17%, while the vast majority of specific associations each receive less than 1% of mentions; this pattern indicates a significant lack of consensus or shared associations regarding Mexican identity, highlighting both ambiguity and fragmentation in collective perceptions.'},
 'DIVERSIDAD, INCLUSIÓN Y PERCEPCIONES DE DESIGUALDAD': {'p25|DEP': 'A strong majority of 67.08% recognize and prefer to speak about the diversity within Mexican cultures, while only 25.33% hold a more unified or singular view of Mexican culture, and 7.58% express no clear opinion, demonstrating widespread acknowledgment of cultural plurality which is essential for designing inclusive cultural, literary, and sports initiatives that reflect Mexico’s rich heritage.'},

In [ ]:
# TODO: preparar documento y gráficas para reporte de salida